# Reactant.jl integration

Reactant.jl is a Julia frontend to the MLIR framework and XLA compiler. It takes Julia code, optimizes aggresively and compiles to CPU / GPU / TPU / distributed clusters.

In [1]:
using Tangles
using Reactant
using Adapt
using Random
using Chairmarks

2025-06-18 23:06:32.426196: I external/xla/xla/pjrt/pjrt_api.cc:93] PJRT_Api is set for device type tpu


In [2]:
Random.seed!(1234)
tn = rand(TensorNetwork, 4, 3)

GenericTensorNetwork (#tensors=4, #inds=6)

Reactant.jl traces the use of its own array types; i.e. it only tracks the operations performed on `Reactant.ConcreteRArray`s.

So first, the arrays of tensors in the Tensor Network need to be converted to `ConcreteRArray`. This is easily done with `Adapt.adapt`.

In [3]:
tnre = adapt(ConcreteRArray, tn)

GenericTensorNetwork (#tensors=4, #inds=6)

In [4]:
@info "before" typeof(tensors(tn)[1])
@info "after" typeof(tensors(tnre)[1])

┌ Info: before
│   typeof((tensors(tn))[1]) = Tensor{Float64, 2, Matrix{Float64}}
└ @ Main /Users/mofeing/Developer/Tangles.jl/examples/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:1
┌ Info: after
│   typeof((tensors(tnre))[1]) = Tensor{Float64, 2, ConcretePJRTArray{Float64, 2, 1, Reactant.Sharding.ShardInfo{Reactant.Sharding.NoSharding, Nothing}}}
└ @ Main /Users/mofeing/Developer/Tangles.jl/examples/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:2


`Reactant.compile` compiles the given code using the compilers mentioned above and returns a _thunk_ (a _functor_, a callable object). The thunk accepts the same type of arguments as the ones passed to `Reactant.compile`.

In [5]:
f = Reactant.compile(contract, (tnre,))
f(tnre)

┌ Warning: Reactant doesn't support sampling of UInt128 with the current interpreter. Falling back to native interpreter.
└ @ Reactant /Users/mofeing/.julia/packages/Reactant/hBRjS/src/Overlay.jl:78


0-dimensional Tensor{Float64, 0, ConcretePJRTArray{Float64, 0, 1, Reactant.Sharding.ShardInfo{Reactant.Sharding.NoSharding, Nothing}}}:
1291.8407836130243

`@compile` is a helper macro for `Reactant.compile`.

In [6]:
f = @compile contract(tnre)
f(tnre)

0-dimensional Tensor{Float64, 0, ConcretePJRTArray{Float64, 0, 1, Reactant.Sharding.ShardInfo{Reactant.Sharding.NoSharding, Nothing}}}:
1291.8407836130243

`@jit` is a helper macro that is equivalent to `@compile` plus directly calling the compiled function.

In [7]:
@jit contract(tnre)

0-dimensional Tensor{Float64, 0, ConcretePJRTArray{Float64, 0, 1, Reactant.Sharding.ShardInfo{Reactant.Sharding.NoSharding, Nothing}}}:
1291.8407836130243

One important aspect is that since allocations are being performed on the "C-side", the allocations shown when profiling are not really representative of the real allocations (only of the allocations being performed on the Julia side).

The time though, should be ok (unless you use asynchronous execution).

In [8]:
@b f(tnre)

9.667 μs (33 allocs: 832 bytes)

## Inspecting what's happenning under the hood

Just like with Julia's `@code_lowered`, `@code_llvm`, `@code_native`, ... introspection tools, Reactant.jl provides the `@code_hlo`, `@code_mhlo` and `@code_xla` tools to inspect the emitted MLIR and HLO code.

Despite its name, `@code_hlo` prints the MLIR representation of the code, which is the main representation Reactant.jl works with. For example, the MLIR code of the compiled contraction is:

In [9]:
@code_hlo contract(tnre)

module @reactant_contract attributes {mhlo.num_partitions = 1 : i64, mhlo.num_replicas = 1 : i64} {
  func.func @main(%arg0: tensor<6x7xf64>, %arg1: tensor<7x6x7x8x5x2xf64>, %arg2: tensor<2x7xf64>, %arg3: tensor<5x8xf64>) -> tensor<f64> {
    %0 = stablehlo.transpose %arg1, dims = [5, 4, 3, 2, 1, 0] : (tensor<7x6x7x8x5x2xf64>) -> tensor<2x5x8x7x6x7xf64>
    %1 = stablehlo.dot_general %arg0, %0, contracting_dims = [1, 0] x [3, 4], precision = [DEFAULT, DEFAULT] : (tensor<6x7xf64>, tensor<2x5x8x7x6x7xf64>) -> tensor<2x5x8x7xf64>
    %2 = stablehlo.dot_general %1, %arg3, contracting_dims = [1, 2] x [0, 1], precision = [DEFAULT, DEFAULT] : (tensor<2x5x8x7xf64>, tensor<5x8xf64>) -> tensor<2x7xf64>
    %3 = stablehlo.dot_general %arg2, %2, contracting_dims = [1, 0] x [1, 0], precision = [DEFAULT, DEFAULT] : (tensor<2x7xf64>, tensor<2x7xf64>) -> tensor<f64>
    return %3 : tensor<f64>
  }
}